# CICIoT2023 — Reproducible 3-Class Model Training (22 Features)

This notebook trains and compares four classifiers for **Benign / DDoS / DoS** traffic using CICIoT2023 pre-extracted CSV features.

## Model input contract

- 18 selected base features.
- 4 deterministic engineered features (`bytes_per_sec`, `syn_ack_ratio`, `hdr_pkt_ratio`, `pkt_cv`).
- 22 ordered model features in total.
- `Number`, `Weight`, `IAT`, `Covariance`, `Radius`, `Magnitue`, and `Variance` are deliberately excluded.

## Important scope and deployment notes

1. This notebook uses **pre-extracted CSV files**, not PCAP files. It cannot repair an inconsistent packet-windowing policy that was applied before these CSVs were generated. Confirm the source extraction procedure and use one label-independent packet-window rule for every class before claiming live-PCAP compatibility.
2. The supplied CSV splits are preserved. Exact duplicate feature rows are removed within/across splits, and feature vectors associated with conflicting labels are excluded. The CSVs do not contain capture IDs, so this cannot guarantee capture/session-level independence. Prefer group-based splits from original PCAP/capture metadata for a stronger generalization test.
3. This is a **three-class** model. All attack families other than `BenignTraffic`, `DDoS-*`, and `DoS-*` are intentionally excluded.
4. CICIoT2023's `Duration` feature represents **TTL**, not elapsed flow duration. `flow_duration` is the separate flow-duration feature.
5. All models are trained with the same 22-feature input contract. The final cell saves each model and its metadata separately.

In [1]:
import os
import gc
import json
import time
import warnings
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib
import sklearn
import imblearn
import xgboost

from packaging.version import Version
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
TARGET_CLASSES = ["Benign", "DDoS", "DoS"]
DATASET_SLUG = "himadri07/ciciot2023"

# Set this manually if the dataset is already extracted somewhere else.
# Example: DATASET_ROOT = Path("/kaggle/input/himadri07/ciciot2023/CICIOT23")
DATASET_ROOT = None

OUTPUT_DIR = Path("./ciciot23_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# XGBoost defaults to CPU for portability. Set True to request CUDA.
# CUDA request uses XGBoost 2.x+; if GPU training fails, the cell retries on CPU.
XGB_PREFER_GPU = True

# Set True only after verifying that the source CSVs were generated with one
# label-independent packet-window policy for every target class.
CSV_WINDOWING_POLICY_CONFIRMED = False
CSV_WINDOWING_POLICY_DESCRIPTION = (
    "Unverified from the pre-extracted CSVs; check the source feature-extraction procedure."
)
if not CSV_WINDOWING_POLICY_CONFIRMED:
    warnings.warn(
        "Packet-window provenance is unverified. Results describe this CSV dataset; "
        "do not claim real-time PCAP compatibility until a uniform, label-independent "
        "windowing policy is verified or the model is retrained from PCAPs."
    )

# Kernel SVM can consume substantial RAM. Set None to use all training rows.
# A stratified sample is used only for SVM when the training set is larger.
SVM_MAX_TRAIN_ROWS = 200_000
SVM_NYSTROEM_COMPONENTS = 250

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Output directory:", OUTPUT_DIR.resolve())


Python: 3.13.16
NumPy: 2.1.3
Pandas: 2.3.3
scikit-learn: 1.6.1
imbalanced-learn: 0.14.2
XGBoost: 3.4.1
Output directory: /kaggle/working/ciciot23_artifacts


/tmp/ipykernel_49/868212705.py:62: UserWarning: Packet-window provenance is unverified. Results describe this CSV dataset; do not claim real-time PCAP compatibility until a uniform, label-independent windowing policy is verified or the model is retrained from PCAPs.
  warnings.warn(


In [2]:

BASE_FEATURES = [
    "flow_duration", "Rate", "Header_Length", "Duration",
    "fin_flag_number", "syn_flag_number", "rst_flag_number",
    "psh_flag_number", "ack_flag_number",
    "ack_count", "syn_count", "fin_count", "rst_count",
    "TCP", "UDP", "ICMP", "AVG", "Std",
]

ENGINEERED_FEATURES = [
    "bytes_per_sec",
    "syn_ack_ratio",
    "hdr_pkt_ratio",
    "pkt_cv",
]

MODEL_FEATURES = BASE_FEATURES + ENGINEERED_FEATURES
EPSILON = 1e-5
FEATURE_ENGINEERING_VERSION = "ciciot22f-v1"

assert len(BASE_FEATURES) == 18
assert len(ENGINEERED_FEATURES) == 4
assert len(MODEL_FEATURES) == 22
assert len(set(MODEL_FEATURES)) == 22


def engineer_features(base_df: pd.DataFrame) -> pd.DataFrame:
    """Build the exact 22-column model matrix from the 18 base features.

    Input must contain the 18 BASE_FEATURES. Output is always ordered as
    MODEL_FEATURES. This function is also the contract the inference service
    must reproduce before calling a saved model.
    """
    missing = [c for c in BASE_FEATURES if c not in base_df.columns]
    if missing:
        raise ValueError(f"Missing base features: {missing}")

    out = base_df.loc[:, BASE_FEATURES].copy()

    # Keep the formulas used in the existing experiments.
    avg = out["AVG"].astype(np.float64)
    rate = out["Rate"].astype(np.float64)

    out["bytes_per_sec"] = rate * avg
    out["syn_ack_ratio"] = (
        out["syn_count"].astype(np.float64) + 1.0
    ) / (out["ack_count"].astype(np.float64) + 1.0)
    out["hdr_pkt_ratio"] = (
        out["Header_Length"].astype(np.float64) / (avg + EPSILON)
    )
    out["pkt_cv"] = out["Std"].astype(np.float64) / (avg + EPSILON)

    out = out.loc[:, MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)
    return out.astype(np.float32)


print("Base features:", len(BASE_FEATURES))
print("Engineered features:", len(ENGINEERED_FEATURES))
print("Final model features:", len(MODEL_FEATURES))
print("Feature contract:", MODEL_FEATURES)


Base features: 18
Engineered features: 4
Final model features: 22
Feature contract: ['flow_duration', 'Rate', 'Header_Length', 'Duration', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'TCP', 'UDP', 'ICMP', 'AVG', 'Std', 'bytes_per_sec', 'syn_ack_ratio', 'hdr_pkt_ratio', 'pkt_cv']


In [3]:

def find_dataset_root():
    if DATASET_ROOT is not None:
        root = Path(DATASET_ROOT).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"DATASET_ROOT does not exist: {root}")
        return root

    common_roots = [
        Path("/kaggle/input/ciciot2023/CICIOT23"),
        Path("/kaggle/input/himadri07/ciciot2023/CICIOT23"),
        Path("/kaggle/input/ciciot2023"),
        Path("/kaggle/input/himadri07/ciciot2023"),
    ]
    for root in common_roots:
        if root.exists() and any(root.rglob("train.csv")):
            return root

    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(DATASET_SLUG))
        if downloaded.exists():
            return downloaded
    except Exception as exc:
        raise RuntimeError(
            "Could not locate the dataset locally and kagglehub download failed. "
            "Set DATASET_ROOT manually or configure Kaggle credentials."
        ) from exc

    raise FileNotFoundError("CICIoT2023 dataset root could not be located.")


def find_one_split_file(root: Path, split_folder: str, filename: str) -> Path:
    expected = root / split_folder / filename
    if expected.exists():
        return expected

    candidates = list(root.rglob(filename))
    candidates = [p for p in candidates if p.parent.name.lower() == split_folder.lower()]
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(f"Could not find {split_folder}/{filename} below {root}")
    raise RuntimeError(
        f"Found multiple candidates for {split_folder}/{filename}: {candidates}. "
        "Set DATASET_ROOT to the exact dataset directory."
    )


base_dir = find_dataset_root()
train_path = find_one_split_file(base_dir, "train", "train.csv")
val_path = find_one_split_file(base_dir, "validation", "validation.csv")
test_path = find_one_split_file(base_dir, "test", "test.csv")

for split_name, path in [("TRAIN", train_path), ("VALIDATION", val_path), ("TEST", test_path)]:
    print(f"{split_name:10s}: {path}")
    assert path.exists()


TRAIN     : /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/train/train.csv
VALIDATION: /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/validation/validation.csv
TEST      : /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/test/test.csv


In [4]:

def map_attack_class(label_series: pd.Series) -> pd.Series:
    labels = label_series.astype(str)
    mapped = pd.Series(index=label_series.index, dtype="object")
    mapped.loc[labels.eq("BenignTraffic")] = "Benign"
    mapped.loc[labels.str.startswith("DDoS-", na=False)] = "DDoS"
    mapped.loc[labels.str.startswith("DoS-", na=False)] = "DoS"
    return mapped


def load_and_clean_split(csv_path: Path, split_name: str) -> pd.DataFrame:
    required_columns = BASE_FEATURES + ["label"]
    header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    missing = [c for c in required_columns if c not in header]
    if missing:
        raise ValueError(f"{split_name}: missing required CSV columns: {missing}")

    dtypes = {col: np.float32 for col in BASE_FEATURES}
    df = pd.read_csv(
        csv_path,
        usecols=required_columns,
        dtype=dtypes,
        low_memory=False,
    )
    raw_rows = len(df)

    df["attack_class"] = map_attack_class(df["label"])
    label_counts = df["attack_class"].value_counts(dropna=False)
    excluded_rows = int(df["attack_class"].isna().sum())
    df = df.drop(columns=["label"])
    df = df.dropna(subset=["attack_class"]).copy()

    # Treat infinities as invalid feature values. Rows with invalid required
    # base features are excluded rather than silently imputed.
    df.loc[:, BASE_FEATURES] = df.loc[:, BASE_FEATURES].replace([np.inf, -np.inf], np.nan)
    before_valid = len(df)
    df = df.dropna(subset=BASE_FEATURES).copy()
    invalid_rows = before_valid - len(df)

    # Negative flow duration and packet rate are not physically meaningful.
    # Duration is TTL in this dataset; do not interpret it as elapsed duration.
    df["flow_duration"] = df["flow_duration"].clip(lower=0)
    df["Rate"] = df["Rate"].clip(lower=0)

    df = df.reset_index(drop=True)
    print(
        f"[{split_name}] raw={raw_rows:,}; excluded_other_classes={excluded_rows:,}; "
        f"invalid_base_rows={invalid_rows:,}; retained={len(df):,}"
    )
    print(f"[{split_name}] retained class counts:\n{df['attack_class'].value_counts().to_string()}")
    return df


train_df = load_and_clean_split(train_path, "TRAIN")
val_df = load_and_clean_split(val_path, "VALIDATION")
test_df = load_and_clean_split(test_path, "TEST")
gc.collect()


[TRAIN] raw=5,491,971; excluded_other_classes=413,277; invalid_base_rows=0; retained=5,078,694
[TRAIN] retained class counts:
attack_class
DDoS      3998500
DoS        950656
Benign     129538
[VALIDATION] raw=1,176,851; excluded_other_classes=88,359; invalid_base_rows=0; retained=1,088,492
[VALIDATION] retained class counts:
attack_class
DDoS      857523
DoS       203450
Benign     27519
[TEST] raw=1,176,851; excluded_other_classes=88,916; invalid_base_rows=0; retained=1,087,935
[TEST] retained class counts:
attack_class
DDoS      855981
DoS       204245
Benign     27709


94

In [5]:

def add_row_fingerprint(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    # Fingerprint only the 18 base features. Engineered features are deterministic
    # functions of these columns and must not be used as a separate identity.
    result["_row_hash"] = pd.util.hash_pandas_object(
        result.loc[:, BASE_FEATURES], index=False
    ).astype("uint64")
    return result


train_df = add_row_fingerprint(train_df)
val_df = add_row_fingerprint(val_df)
test_df = add_row_fingerprint(test_df)

# If an identical feature vector is assigned different labels anywhere across
# the supplied splits, remove that feature vector from every split.
all_hash_labels = pd.concat(
    [
        train_df[["_row_hash", "attack_class"]],
        val_df[["_row_hash", "attack_class"]],
        test_df[["_row_hash", "attack_class"]],
    ],
    ignore_index=True,
)
conflicting_hashes = set(
    all_hash_labels.groupby("_row_hash")["attack_class"].nunique().loc[lambda s: s > 1].index
)
print(f"Conflicting feature fingerprints removed from all splits: {len(conflicting_hashes):,}")

if conflicting_hashes:
    train_df = train_df[~train_df["_row_hash"].isin(conflicting_hashes)].copy()
    val_df = val_df[~val_df["_row_hash"].isin(conflicting_hashes)].copy()
    test_df = test_df[~test_df["_row_hash"].isin(conflicting_hashes)].copy()

# Keep one copy of an identical feature vector within each split.
for name, frame in [("TRAIN", train_df), ("VALIDATION", val_df), ("TEST", test_df)]:
    before = len(frame)
    frame.drop_duplicates(subset="_row_hash", keep="first", inplace=True)
    frame.reset_index(drop=True, inplace=True)
    print(f"[{name}] within-split duplicate feature rows removed: {before - len(frame):,}")

# Preserve training rows. Remove exact feature-vector overlap from validation,
# then remove any remaining train/validation overlap from test.
train_hashes = set(train_df["_row_hash"].tolist())
val_before = len(val_df)
val_df = val_df[~val_df["_row_hash"].isin(train_hashes)].reset_index(drop=True)
print(f"Validation rows overlapping train removed: {val_before - len(val_df):,}")

train_val_hashes = train_hashes.union(set(val_df["_row_hash"].tolist()))
test_before = len(test_df)
test_df = test_df[~test_df["_row_hash"].isin(train_val_hashes)].reset_index(drop=True)
print(f"Test rows overlapping train/validation removed: {test_before - len(test_df):,}")

for split_name, frame in [("TRAIN", train_df), ("VALIDATION", val_df), ("TEST", test_df)]:
    counts = frame["attack_class"].value_counts()
    print(f"\n{split_name} final rows: {len(frame):,}")
    print(counts.reindex(TARGET_CLASSES, fill_value=0))

# Training must contain all three target classes for multiclass training.
missing_train_classes = sorted(set(TARGET_CLASSES) - set(train_df["attack_class"].unique()))
if missing_train_classes:
    raise ValueError(f"Training split is missing target classes: {missing_train_classes}")

# Remove the temporary fingerprint before feature construction.
for frame in (train_df, val_df, test_df):
    frame.drop(columns=["_row_hash"], inplace=True)

gc.collect()


Conflicting feature fingerprints removed from all splits: 13,219
[TRAIN] within-split duplicate feature rows removed: 1,353,366
[VALIDATION] within-split duplicate feature rows removed: 133,740
[TEST] within-split duplicate feature rows removed: 132,808
Validation rows overlapping train removed: 287,103
Test rows overlapping train/validation removed: 309,196

TRAIN final rows: 3,648,198
attack_class
Benign     126388
DDoS      2692980
DoS        828830
Name: count, dtype: int64

VALIDATION final rows: 651,089
attack_class
Benign     26042
DDoS      463349
DoS       161698
Name: count, dtype: int64

TEST final rows: 629,421
attack_class
Benign     25940
DDoS      444745
DoS       158736
Name: count, dtype: int64


7

In [6]:

def make_xy(frame: pd.DataFrame, split_name: str):
    X = engineer_features(frame.loc[:, BASE_FEATURES])
    y = frame["attack_class"].astype(str).reset_index(drop=True)

    finite_mask = np.isfinite(X.to_numpy(dtype=np.float64)).all(axis=1)
    invalid_engineered = int((~finite_mask).sum())
    if invalid_engineered:
        X = X.loc[finite_mask].reset_index(drop=True)
        y = y.loc[finite_mask].reset_index(drop=True)

    print(f"[{split_name}] X shape={X.shape}; invalid engineered rows removed={invalid_engineered:,}")
    if list(X.columns) != MODEL_FEATURES:
        raise AssertionError("Feature column order does not match MODEL_FEATURES")
    return X, y


X_train, y_train = make_xy(train_df, "TRAIN")
X_val, y_val = make_xy(val_df, "VALIDATION")
X_test, y_test = make_xy(test_df, "TEST")

for split_name, y in [("TRAIN", y_train), ("VALIDATION", y_val), ("TEST", y_test)]:
    print(f"\n{split_name} class distribution:")
    print(y.value_counts().reindex(TARGET_CLASSES, fill_value=0))

if y_train.nunique() != len(TARGET_CLASSES):
    raise ValueError("Training data does not contain all three classes after feature construction.")

assert X_train.shape[1] == 22
assert X_val.shape[1] == 22
assert X_test.shape[1] == 22
assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_val.to_numpy()).all()
assert np.isfinite(X_test.to_numpy()).all()


[TRAIN] X shape=(3648198, 22); invalid engineered rows removed=0
[VALIDATION] X shape=(651089, 22); invalid engineered rows removed=0
[TEST] X shape=(629421, 22); invalid engineered rows removed=0

TRAIN class distribution:
attack_class
Benign     126388
DDoS      2692980
DoS        828830
Name: count, dtype: int64

VALIDATION class distribution:
attack_class
Benign     26042
DDoS      463349
DoS       161698
Name: count, dtype: int64

TEST class distribution:
attack_class
Benign     25940
DDoS      444745
DoS       158736
Name: count, dtype: int64


In [7]:
print("Ordered model feature contract:")
for index, feature in enumerate(MODEL_FEATURES, start=1):
    print(f"{index:02d}. {feature}")

print("\nFeature dtypes:")
print(X_train.dtypes.value_counts())

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).reindex(TARGET_CLASSES))

print("\nValidation class proportions:")
print(y_val.value_counts(normalize=True).reindex(TARGET_CLASSES))

print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).reindex(TARGET_CLASSES))

# Reminder: the split CSV files do not expose capture/session identifiers.
print(
    "\nLeakage caveat: exact duplicate feature rows have been purged, but "
    "capture/session-level separation cannot be verified from these CSVs alone."
)


Ordered model feature contract:
01. flow_duration
02. Rate
03. Header_Length
04. Duration
05. fin_flag_number
06. syn_flag_number
07. rst_flag_number
08. psh_flag_number
09. ack_flag_number
10. ack_count
11. syn_count
12. fin_count
13. rst_count
14. TCP
15. UDP
16. ICMP
17. AVG
18. Std
19. bytes_per_sec
20. syn_ack_ratio
21. hdr_pkt_ratio
22. pkt_cv

Feature dtypes:
float32    22
Name: count, dtype: int64

Training class proportions:
attack_class
Benign    0.034644
DDoS      0.738167
DoS       0.227189
Name: proportion, dtype: float64

Validation class proportions:
attack_class
Benign    0.039998
DDoS      0.711652
DoS       0.248350
Name: proportion, dtype: float64

Test class proportions:
attack_class
Benign    0.041212
DDoS      0.706594
DoS       0.252194
Name: proportion, dtype: float64

Leakage caveat: exact duplicate feature rows have been purged, but capture/session-level separation cannot be verified from these CSVs alone.


In [8]:
brf_model = BalancedRandomForestClassifier(
    n_estimators=250,
    max_depth=14,
    min_samples_leaf=20,
    max_features="sqrt",
    sampling_strategy="all",
    replacement=True,
    n_jobs=-1,
    random_state=SEED,
)

print("Training Balanced Random Forest on 22 features...")
brf_started = time.time()
brf_model.fit(X_train, y_train)
brf_train_seconds = time.time() - brf_started
print(f"Balanced Random Forest training complete in {brf_train_seconds / 60:.2f} minutes.")


Training Balanced Random Forest on 22 features...
Balanced Random Forest training complete in 3.35 minutes.


In [9]:
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

# Apply the same square-root balanced-weight policy to both training and
# validation evaluation weights. This avoids the previous train/validation
# weighting mismatch.
train_sample_weights = np.sqrt(
    compute_sample_weight(class_weight="balanced", y=y_train_enc)
)
val_sample_weights = np.sqrt(
    compute_sample_weight(class_weight="balanced", y=y_val_enc)
)


def build_xgb(device: str):
    params = dict(
        n_estimators=800,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=20,
        reg_alpha=1.0,
        reg_lambda=5.0,
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
        early_stopping_rounds=40,
    )
    if device == "cuda":
        if Version(xgboost.__version__) >= Version("2.0.0"):
            params["device"] = "cuda"
        else:
            params["tree_method"] = "gpu_hist"
    return XGBClassifier(**params)


requested_device = "cuda" if XGB_PREFER_GPU else "cpu"
print(f"Training XGBoost (requested device: {requested_device})...")
xgb_started = time.time()
xgb_model = build_xgb(requested_device)
try:
    xgb_model.fit(
        X_train,
        y_train_enc,
        sample_weight=train_sample_weights,
        eval_set=[(X_val, y_val_enc)],
        sample_weight_eval_set=[val_sample_weights],
        verbose=False,
    )
except Exception as exc:
    if requested_device != "cuda":
        raise
    warnings.warn(f"GPU XGBoost training failed; retrying on CPU. Reason: {exc}")
    xgb_model = build_xgb("cpu")
    xgb_model.fit(
        X_train,
        y_train_enc,
        sample_weight=train_sample_weights,
        eval_set=[(X_val, y_val_enc)],
        sample_weight_eval_set=[val_sample_weights],
        verbose=False,
    )

xgb_train_seconds = time.time() - xgb_started
print(f"XGBoost training complete in {xgb_train_seconds / 60:.2f} minutes.")
print("XGBoost best iteration:", getattr(xgb_model, "best_iteration", None))
print("XGBoost classes:", list(label_encoder.classes_))


Training XGBoost (requested device: cpu)...
XGBoost training complete in 9.08 minutes.
XGBoost best iteration: 799
XGBoost classes: ['Benign', 'DDoS', 'DoS']


In [10]:
# Calibration is wrapped around the ENTIRE preprocessing/model pipeline.
# Thus StandardScaler and Nystroem are refitted inside each calibration fold,
# avoiding the preprocessing leakage in the earlier notebook.
svm_base_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rbf_map", Nystroem(
        kernel="rbf",
        gamma=0.1,
        n_components=SVM_NYSTROEM_COMPONENTS,
        random_state=SEED,
    )),
    ("classifier", SGDClassifier(
        loss="hinge",
        penalty="l2",
        alpha=1e-4,
        max_iter=1000,
        tol=1e-3,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )),
])

svm_model = CalibratedClassifierCV(
    estimator=svm_base_pipeline,
    method="sigmoid",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    n_jobs=-1,
)

if SVM_MAX_TRAIN_ROWS is not None and len(X_train) > SVM_MAX_TRAIN_ROWS:
    X_svm_train, _, y_svm_train, _ = train_test_split(
        X_train,
        y_train,
        train_size=SVM_MAX_TRAIN_ROWS,
        stratify=y_train,
        random_state=SEED,
    )
    print(f"SVM uses a stratified training subset: {len(X_svm_train):,} / {len(X_train):,} rows")
else:
    X_svm_train, y_svm_train = X_train, y_train
    print(f"SVM uses all training rows: {len(X_svm_train):,}")

print("Training calibrated SVM...")
svm_started = time.time()
svm_model.fit(X_svm_train, y_svm_train)
svm_train_seconds = time.time() - svm_started
print(f"SVM training complete in {svm_train_seconds / 60:.2f} minutes.")


SVM uses a stratified training subset: 200,000 / 3,648,198 rows
Training calibrated SVM...
SVM training complete in 0.12 minutes.


In [11]:
logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1500,
        class_weight="balanced",
        random_state=SEED,
    )),
])

print("Training Logistic Regression on 22 features...")
logreg_started = time.time()
logreg_model.fit(X_train, y_train)
logreg_train_seconds = time.time() - logreg_started
print(f"Logistic Regression training complete in {logreg_train_seconds / 60:.2f} minutes.")

trained_models = {
    "balanced_random_forest": brf_model,
    "xgboost": xgb_model,
    "calibrated_svm": svm_model,
    "logistic_regression": logreg_model,
}
training_times_seconds = {
    "balanced_random_forest": brf_train_seconds,
    "xgboost": xgb_train_seconds,
    "calibrated_svm": svm_train_seconds,
    "logistic_regression": logreg_train_seconds,
}


Training Logistic Regression on 22 features...
Logistic Regression training complete in 2.76 minutes.


In [12]:

def predict_model_labels(model_name: str, model, X: pd.DataFrame) -> np.ndarray:
    if model_name == "xgboost":
        predicted_ids = np.asarray(model.predict(X), dtype=int)
        return label_encoder.inverse_transform(predicted_ids)
    return np.asarray(model.predict(X), dtype=object)


def evaluate_one(model_name: str, model, X: pd.DataFrame, y_true: pd.Series, split_name: str):
    y_pred = predict_model_labels(model_name, model, X)

    report = classification_report(
        y_true,
        y_pred,
        labels=TARGET_CLASSES,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred, labels=TARGET_CLASSES)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(
            y_true, y_pred, labels=TARGET_CLASSES, average="macro", zero_division=0
        )),
        "macro_recall": float(recall_score(
            y_true, y_pred, labels=TARGET_CLASSES, average="macro", zero_division=0
        )),
        "macro_f1": float(f1_score(
            y_true, y_pred, labels=TARGET_CLASSES, average="macro", zero_division=0
        )),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    print(f"\n===== {model_name.upper()} — {split_name.upper()} =====")
    print(pd.Series(metrics).to_string())
    print("\nClassification report:")
    print(classification_report(
        y_true, y_pred, labels=TARGET_CLASSES, target_names=TARGET_CLASSES, digits=4,
        zero_division=0
    ))
    print("Confusion matrix (rows=true, columns=predicted):")
    print(pd.DataFrame(
        cm,
        index=[f"True_{c}" for c in TARGET_CLASSES],
        columns=[f"Pred_{c}" for c in TARGET_CLASSES],
    ))

    return {
        "metrics": metrics,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }


In [13]:
# Hyperparameters and model families are defined above. Use validation metrics
# for model selection. Test metrics are reported once as final held-out results;
# do not tune parameters in response to test performance.
evaluation_results = {}

for model_name, model in trained_models.items():
    print("\n" + "#" * 90)
    print("Evaluating:", model_name)
    evaluation_results[model_name] = {
        "validation": evaluate_one(model_name, model, X_val, y_val, "validation"),
        "test": evaluate_one(model_name, model, X_test, y_test, "test"),
    }

print("\nAll model evaluations complete.")



##########################################################################################
Evaluating: balanced_random_forest

===== BALANCED_RANDOM_FOREST — VALIDATION =====
accuracy             0.847887
balanced_accuracy    0.891804
macro_precision      0.831267
macro_recall         0.891804
macro_f1             0.856510
weighted_f1          0.852772

Classification report:
              precision    recall  f1-score   support

      Benign     0.8983    0.9987    0.9458     26042
        DDoS     0.9369    0.8451    0.8886    463349
         DoS     0.6586    0.8317    0.7351    161698

    accuracy                         0.8479    651089
   macro avg     0.8313    0.8918    0.8565    651089
weighted avg     0.8662    0.8479    0.8528    651089

Confusion matrix (rows=true, columns=predicted):
             Pred_Benign  Pred_DDoS  Pred_DoS
True_Benign        26007          4        31
True_DDoS           2093     391560     69696
True_DoS             850      26365    134483

=====

In [14]:
leaderboard_rows = []
for model_name in trained_models:
    val_metrics = evaluation_results[model_name]["validation"]["metrics"]
    test_metrics = evaluation_results[model_name]["test"]["metrics"]
    leaderboard_rows.append({
        "model": model_name,
        "validation_macro_f1": val_metrics["macro_f1"],
        "validation_balanced_accuracy": val_metrics["balanced_accuracy"],
        "validation_macro_recall": val_metrics["macro_recall"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "training_minutes": training_times_seconds[model_name] / 60.0,
    })

leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    by=["validation_macro_f1", "validation_balanced_accuracy"],
    ascending=False,
).reset_index(drop=True)

print(leaderboard.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Select using validation only — never choose the winner based on test metrics.
selected_model_name = leaderboard.iloc[0]["model"]
selected_model = trained_models[selected_model_name]
print(f"\nSelected candidate based on validation Macro-F1: {selected_model_name}")
print("Test metrics are reported for transparency, not used for selection.")


                 model  validation_macro_f1  validation_balanced_accuracy  validation_macro_recall  test_macro_f1  test_balanced_accuracy  training_minutes
               xgboost               0.9013                        0.8995                   0.8995         0.9016                  0.9004            9.0757
balanced_random_forest               0.8565                        0.8918                   0.8918         0.8570                  0.8919            3.3452
        calibrated_svm               0.7769                        0.7906                   0.7906         0.7768                  0.7910            0.1241
   logistic_regression               0.7355                        0.8120                   0.8120         0.7341                  0.8103            2.7636

Selected candidate based on validation Macro-F1: xgboost
Test metrics are reported for transparency, not used for selection.


In [15]:
feature_importances = {}

# Balanced Random Forest impurity-based importance.
feature_importances["balanced_random_forest"] = pd.Series(
    brf_model.feature_importances_, index=MODEL_FEATURES
).sort_values(ascending=False)

# XGBoost built-in feature importance.
feature_importances["xgboost"] = pd.Series(
    xgb_model.feature_importances_, index=MODEL_FEATURES
).sort_values(ascending=False)

# Logistic Regression coefficient magnitude (mean absolute coefficient over classes).
logreg_classifier = logreg_model.named_steps["classifier"]
logreg_importance = np.abs(logreg_classifier.coef_).mean(axis=0)
feature_importances["logistic_regression"] = pd.Series(
    logreg_importance, index=MODEL_FEATURES
).sort_values(ascending=False)

# SVM has no directly comparable tree/coefficient importance. Estimate it using
# permutation importance on a modest validation sample to control runtime.
perm_n = min(3000, len(X_val))
perm_rng = np.random.RandomState(SEED)
perm_indices = perm_rng.choice(len(X_val), size=perm_n, replace=False)
X_perm = X_val.iloc[perm_indices]
y_perm = y_val.iloc[perm_indices]

print(f"Computing SVM permutation importance on {perm_n:,} validation rows...")
svm_perm = permutation_importance(
    svm_model,
    X_perm,
    y_perm,
    scoring="f1_macro",
    n_repeats=2,
    random_state=SEED,
    n_jobs=1,
)
svm_raw_importance = np.maximum(svm_perm.importances_mean, 0.0)
svm_total = svm_raw_importance.sum()
if svm_total > 0:
    svm_raw_importance = svm_raw_importance / svm_total
feature_importances["calibrated_svm"] = pd.Series(
    svm_raw_importance, index=MODEL_FEATURES
).sort_values(ascending=False)

for model_name, importance_series in feature_importances.items():
    print(f"\nTop features — {model_name}:")
    print(importance_series.head(15).to_string(float_format=lambda x: f"{x:.5f}"))

# Benign baseline is calculated from training data only (no validation/test leakage).
benign_train = X_train.loc[y_train.eq("Benign")]
benign_baseline = {}
for feature in MODEL_FEATURES:
    q25 = float(benign_train[feature].quantile(0.25))
    q75 = float(benign_train[feature].quantile(0.75))
    benign_baseline[feature] = {
        "median": float(benign_train[feature].median()),
        "iqr": float(max(q75 - q25, 1e-6)),
        "p95": float(benign_train[feature].quantile(0.95)),
    }

print(f"\nBenign baseline calculated for {len(benign_baseline)} features.")


Computing SVM permutation importance on 3,000 validation rows...

Top features — balanced_random_forest:
rst_count         0.17387
Header_Length     0.12277
AVG               0.11684
flow_duration     0.10251
Std               0.09000
hdr_pkt_ratio     0.06148
pkt_cv            0.05973
Rate              0.04427
bytes_per_sec     0.04172
ack_flag_number   0.03848
Duration          0.03378
syn_ack_ratio     0.01689
UDP               0.01634
ICMP              0.01432
syn_count         0.01330

Top features — xgboost:
fin_flag_number   0.27627
ICMP              0.17524
UDP               0.13647
psh_flag_number   0.11037
rst_count         0.07086
syn_flag_number   0.04311
ack_flag_number   0.03854
TCP               0.03339
flow_duration     0.01716
rst_flag_number   0.01457
AVG               0.01139
Header_Length     0.01101
fin_count         0.01046
ack_count         0.00979
hdr_pkt_ratio     0.00909

Top features — logistic_regression:
flow_duration     6.21938
syn_ack_ratio     4.59488
s

In [16]:
smoke_frame = X_test.copy()
smoke_frame["label"] = y_test.to_numpy()

sample_size = min(500, len(smoke_frame))
if len(smoke_frame) > sample_size:
    smoke_sample, _ = train_test_split(
        smoke_frame,
        train_size=sample_size,
        stratify=smoke_frame["label"],
        random_state=SEED,
    )
else:
    smoke_sample = smoke_frame.copy()

smoke_sample = smoke_sample.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
smoke_path = OUTPUT_DIR / "ciciot23_test_smoke_sample.csv"
smoke_sample.to_csv(smoke_path, index=False)

print(f"Saved inference smoke-test sample: {smoke_path}")
print("Rows:", len(smoke_sample))
print("Columns:", len(smoke_sample.columns), "(22 features + label)")
print(smoke_sample["label"].value_counts().reindex(TARGET_CLASSES, fill_value=0))


Saved inference smoke-test sample: ciciot23_artifacts/ciciot23_test_smoke_sample.csv
Rows: 500
Columns: 23 (22 features + label)
label
Benign     21
DDoS      353
DoS       126
Name: count, dtype: int64


In [17]:
# Every bundle expects a DataFrame with exactly MODEL_FEATURES in this order.
# For raw PCAP inference, the service must first extract BASE_FEATURES and apply
# engineer_features() using the same formulas before passing data to the model.

library_versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "imbalanced_learn": imblearn.__version__,
    "xgboost": xgboost.__version__,
    "joblib": joblib.__version__,
}

feature_engineering_manifest = {
    "version": FEATURE_ENGINEERING_VERSION,
    "base_features": BASE_FEATURES,
    "engineered_features": ENGINEERED_FEATURES,
    "model_features": MODEL_FEATURES,
    "formulas": {
        "bytes_per_sec": "Rate * AVG",
        "syn_ack_ratio": "(syn_count + 1.0) / (ack_count + 1.0)",
        "hdr_pkt_ratio": "Header_Length / (AVG + 1e-5)",
        "pkt_cv": "Std / (AVG + 1e-5)",
    },
    "epsilon": EPSILON,
}

artifact_specs = {
    "balanced_random_forest": {
        "filename": "ciciot23_balanced_random_forest_22f.joblib",
        "model": brf_model,
        "label_encoder": None,
        "training_seconds": brf_train_seconds,
    },
    "xgboost": {
        "filename": "ciciot23_xgboost_22f.joblib",
        "model": xgb_model,
        "label_encoder": label_encoder,
        "training_seconds": xgb_train_seconds,
    },
    "calibrated_svm": {
        "filename": "ciciot23_calibrated_svm_22f.joblib",
        "model": svm_model,
        "label_encoder": None,
        "training_seconds": svm_train_seconds,
    },
    "logistic_regression": {
        "filename": "ciciot23_logistic_regression_22f.joblib",
        "model": logreg_model,
        "label_encoder": None,
        "training_seconds": logreg_train_seconds,
    },
}

saved_artifacts = {}
for model_name, spec in artifact_specs.items():
    bundle = {
        "artifact_format_version": 1,
        "model_name": model_name,
        "model_version": "1.0.0",
        "model": spec["model"],
        "label_encoder": spec["label_encoder"],
        "classes": TARGET_CLASSES,
        "feature_names": MODEL_FEATURES,
        "base_feature_names": BASE_FEATURES,
        "feature_engineering": feature_engineering_manifest,
        "target_mapping": {
            "BenignTraffic": "Benign",
            "labels_starting_with_DDoS_dash": "DDoS",
            "labels_starting_with_DoS_dash": "DoS",
            "all_other_labels": "excluded_during_training",
        },
        "training_scope": {
            "dataset": "CICIoT2023",
            "classification": "3-class",
            "classes": TARGET_CLASSES,
            "input_source": "pre-extracted CSV feature rows",
            "windowing_provenance": CSV_WINDOWING_POLICY_DESCRIPTION,
            "windowing_policy_confirmed": bool(CSV_WINDOWING_POLICY_CONFIRMED),
            "capture_group_split_verified": False,
        },
        "metrics": evaluation_results[model_name],
        "feature_importance": {
            feature: float(value)
            for feature, value in feature_importances[model_name].items()
        },
        "benign_baseline": benign_baseline,
        "training_seconds": float(spec["training_seconds"]),
        "selected_by_validation_macro_f1": bool(model_name == selected_model_name),
        "library_versions": library_versions,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    artifact_path = OUTPUT_DIR / spec["filename"]
    joblib.dump(bundle, artifact_path, compress=3)
    saved_artifacts[model_name] = {
        "path": str(artifact_path.resolve()),
        "size_mb": round(artifact_path.stat().st_size / (1024 ** 2), 2),
    }
    print(f"Saved {model_name}: {artifact_path} ({saved_artifacts[model_name]['size_mb']:.2f} MB)")

# Also save XGBoost in its native JSON representation for portability.
xgb_native_path = OUTPUT_DIR / "ciciot23_xgboost_native.json"
xgb_model.save_model(xgb_native_path)
print(f"Saved native XGBoost model: {xgb_native_path}")

# Save comparison and feature contract files alongside model bundles.
leaderboard_path = OUTPUT_DIR / "model_comparison.csv"
leaderboard.to_csv(leaderboard_path, index=False)

manifest = {
    "selected_candidate": selected_model_name,
    "classes": TARGET_CLASSES,
    "feature_engineering": feature_engineering_manifest,
    "artifacts": saved_artifacts,
    "xgboost_native_model": str(xgb_native_path.resolve()),
    "smoke_test_csv": str(smoke_path.resolve()),
    "model_comparison_csv": str(leaderboard_path.resolve()),
    "library_versions": library_versions,
}
manifest_path = OUTPUT_DIR / "model_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nAll model artifacts saved.")
print("Manifest:", manifest_path.resolve())
print("\nFiles in output directory:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{path.name:48s} {path.stat().st_size / (1024 ** 2):8.2f} MB")


Saved balanced_random_forest: ciciot23_artifacts/ciciot23_balanced_random_forest_22f.joblib (704.71 MB)
Saved xgboost: ciciot23_artifacts/ciciot23_xgboost_22f.joblib (6.84 MB)
Saved calibrated_svm: ciciot23_artifacts/ciciot23_calibrated_svm_22f.joblib (0.73 MB)
Saved logistic_regression: ciciot23_artifacts/ciciot23_logistic_regression_22f.joblib (0.00 MB)
Saved native XGBoost model: ciciot23_artifacts/ciciot23_xgboost_native.json

All model artifacts saved.
Manifest: /kaggle/working/ciciot23_artifacts/model_manifest.json

Files in output directory:
ciciot23_balanced_random_forest_22f.joblib         704.71 MB
ciciot23_calibrated_svm_22f.joblib                   0.73 MB
ciciot23_logistic_regression_22f.joblib              0.00 MB
ciciot23_test_smoke_sample.csv                       0.06 MB
ciciot23_xgboost_22f.joblib                          6.84 MB
ciciot23_xgboost_native.json                        27.09 MB
model_comparison.csv                                 0.00 MB
model_manifest.jso

In [18]:

# ==========================================
# CELL 19: PACKAGE ALL MODEL ARTIFACTS
# ==========================================

import os
import zipfile
from pathlib import Path
from IPython.display import display, FileLink

# Directory containing saved models
OUTPUT_DIR = Path("/kaggle/working")

# ZIP destination
ZIP_PATH = OUTPUT_DIR / "ciciot23_all_model_artifacts.zip"

# Files and directories to include
INCLUDE_EXTENSIONS = {
    ".joblib",
    ".json",
    ".csv",
    ".txt",
    ".pkl",
    ".yaml",
    ".yml"
}

# Exclude the ZIP itself and unrelated files
files_to_zip = [
    f for f in OUTPUT_DIR.rglob("*")
    if f.is_file()
    and f != ZIP_PATH
    and f.suffix.lower() in INCLUDE_EXTENSIONS
]

if not files_to_zip:
    raise FileNotFoundError(
        "No model artifacts found in /kaggle/working. "
        "Run the model-saving cell first."
    )

# Create ZIP
with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for file in files_to_zip:
        arcname = file.relative_to(OUTPUT_DIR)
        zipf.write(file, arcname)

# Display summary
size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)

print("=" * 50)
print("MODEL ARTIFACTS PACKAGED")
print("=" * 50)

for file in files_to_zip:
    print(f"  {file.name}")

print(f"\nTotal files: {len(files_to_zip)}")
print(f"ZIP size: {size_mb:.2f} MB")
print(f"Saved at: {ZIP_PATH}")

# Kaggle download link
display(FileLink(str(ZIP_PATH)))


MODEL ARTIFACTS PACKAGED
  ciciot23_balanced_random_forest_22f.joblib
  ciciot23_xgboost_native.json
  ciciot23_calibrated_svm_22f.joblib
  ciciot23_logistic_regression_22f.joblib
  model_manifest.json
  model_comparison.csv
  ciciot23_test_smoke_sample.csv
  ciciot23_xgboost_22f.joblib

Total files: 8
ZIP size: 716.10 MB
Saved at: /kaggle/working/ciciot23_all_model_artifacts.zip


/kaggle/working/ciciot23_all_model_artifacts.zip